# Agent-to-Agent (A2A) Design Patterns

A practical session covering how multiple AI agents communicate, collaborate,
and coordinate with each other to solve complex software development tasks.

## Patterns Covered

| # | Pattern | Topology | Scenario |
|---|---|---|---|
| 1 | **Handoff** | Linear chain | Feature Request Pipeline |
| 2 | **Supervisor / Orchestrator** | Hub-and-spoke | Multi-Dimension Code Review |
| 3 | **Peer Debate / Collaboration** | Round-robin + judge | Architecture Decision |
| 4 | **Critic / Reflection** | Feedback loop | Iterative Code Improvement |
| 5 | **Agent as a Tool** | Nested agent loops | Software Delivery Orchestrator |

## A2A vs Single-Agent Patterns

| Dimension | Single-Agent Patterns | Agent-to-Agent (A2A) |
|---|---|---|
| **Number of LLMs** | One LLM, many tool calls | Multiple LLMs, each with a distinct role |
| **Control flow** | Developer or one LLM decides | Agents hand off to each other |
| **System prompts** | One persona | Different persona per agent |
| **Perspectives** | Single viewpoint | Multiple, potentially conflicting views |
| **Parallelism** | Tool calls in parallel | Entire agents in parallel |
| **Best for** | Well-scoped tasks | Complex tasks needing specialist depth |

> **Domain:** All examples use software development scenarios.

In [ ]:
import os
import json
import time
import concurrent.futures
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
client = OpenAI()

MODEL = "gpt-4.1-mini"

# Shared helper used by all patterns
def llm_call(system: str, user: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user}
        ]
    )
    return response.choices[0].message.content

print("Setup complete. Model:", MODEL)

---
## A2A Communication — Core Concepts

In A2A systems, agents are not just tools — each agent is an autonomous LLM
with its own **system prompt (persona)**, **input contract**, and **output contract**.

### What passes between agents?

The output of Agent A becomes the structured input to Agent B.
Each agent must produce output the next agent can reliably consume:

| Output type | Used when |
|---|---|
| Plain text narrative | Next agent is a reviewer or summariser |
| Structured JSON | Next agent needs to parse specific fields |
| Code block | Next agent writes tests or reviews implementation |
| Score + feedback | Next agent is a generator that self-corrects |

### Communication Topologies

```
HANDOFF (linear chain):
  Agent A ──► Agent B ──► Agent C ──► Output

SUPERVISOR (hub-and-spoke):
        ┌──► Worker A ──┐
Input ──┤  Supervisor   ├──► Synthesised Output
        └──► Worker B ──┘

PEER DEBATE (round-robin + judge):
  Agent A proposes ──► Agent B critiques ──► Agent A rebuts
  Agent B proposes ──► Agent A critiques ──► Agent B rebuts
                                              └──► Judge decides

CRITIC / REFLECTION (loop):
  Generator ──► output ──► Critic ──► feedback ──► Generator (improved)
       ▲                                                    │
       └──────────────── repeat until score >= 8 ◄──────────┘
```

### Key Design Rules
1. Each agent's system prompt defines its **role, constraints, and output format**
2. Pass only what the next agent needs — trim or structure the handoff payload
3. Use `concurrent.futures.ThreadPoolExecutor` when agents can run in parallel
4. Always have a **stop condition** to prevent infinite loops

---
# Pattern 1 — Handoff

## Concept

Handoff is the simplest A2A pattern: agents are arranged in a **linear pipeline**
where each agent completes its specialised task and hands its output to the next.
Control flows in one direction only — no loops, no branching.

```
Feature Request
      │
      ▼
[Requirements Analyst]  ← structures raw ideas into acceptance criteria
      │  requirements doc
      ▼
[Software Architect]    ← designs the technical solution
      │  technical design
      ▼
[Code Generator]        ← implements based on the design
      │  Python code
      ▼
[QA Test Writer]        ← writes pytest tests for the implementation
      │
      ▼
   Deliverables
```

## Why This Is A2A

Each agent receives the **specific output of the previous agent** — not the original user input.
The Architect never reads the raw feature request — it only reads the Analyst's requirements doc.
The QA Writer never reads the design — it only reads the code. Each handoff narrows
and transforms the payload for the specialist receiving it.

## When to Use
- Tasks with a natural professional pipeline (analyst → architect → developer → tester)
- When each phase genuinely needs only the previous phase's output
- When intermediate artefacts have standalone value

## Agents

| Agent | Input | Output |
|---|---|---|
| Requirements Analyst | Raw feature description | Numbered acceptance criteria doc |
| Software Architect | Requirements doc | Component design + API contract |
| Code Generator | Technical design | Python implementation |
| QA Test Writer | Implementation code | pytest test suite |

In [ ]:
# ── Pattern 1: Handoff — Feature Request Pipeline ────────────────────────────

ANALYST_SYSTEM = """You are a Senior Requirements Analyst at a software company.
Take the raw feature request and produce a structured requirements document.

Output format (use exactly these headers):
## Feature: <title>
**Goal:** <one sentence>
**Acceptance Criteria:**
1. <criterion — specific and testable>
2. <criterion>
...
**Out of Scope:** <what is explicitly excluded>
**Dependencies:** <known technical dependencies>

Produce 4–6 acceptance criteria. Be specific, testable, and concise."""

ARCHITECT_SYSTEM = """You are a Senior Software Architect.
Receive a requirements document and produce a technical design.

Output format (use exactly these headers):
## Technical Design: <feature title>
**Architecture Decision:** <chosen approach and rationale in 2 sentences>
**Components:**
- `<ClassName>`: <responsibility>
**Data Model:** <key data structures or fields as a short list>
**API Contract:** <function signatures or REST endpoints>
**Key Risks:** <top 2 technical risks>

Output only the design document, no preamble."""

CODE_GEN_SYSTEM = """You are a Senior Python Developer.
Receive a technical design document and implement it.
- Write clean, production-quality Python with type hints
- Follow the component names and API contract from the design exactly
- Include docstrings on public functions and classes
- Use stdlib only unless the design explicitly specifies a library
- Output ONLY the Python code — no explanation before or after"""

QA_SYSTEM = """You are a Senior QA Engineer specialising in Python.
Receive implementation code and write a comprehensive pytest test suite.
- Cover happy path, edge cases, and error cases
- Use pytest fixtures where appropriate
- Mock external dependencies with unittest.mock
- Each test function must have a clear docstring describing what it tests
- Output ONLY the pytest code — no explanation before or after"""


def run_analyst(feature_request: str) -> str:
    return llm_call(ANALYST_SYSTEM, f"Feature request:\n{feature_request}")

def run_architect(requirements: str) -> str:
    return llm_call(ARCHITECT_SYSTEM, f"Requirements document:\n{requirements}")

def run_code_generator(design: str) -> str:
    return llm_call(CODE_GEN_SYSTEM, f"Technical design:\n{design}")

def run_qa_writer(code: str) -> str:
    return llm_call(QA_SYSTEM, f"Implementation code:\n{code}")


def run_handoff_pipeline(feature_request: str) -> tuple[str, str, str, str]:
    """Execute the four-agent handoff pipeline sequentially."""
    requirements = run_analyst(feature_request)        # Agent 1
    design       = run_architect(requirements)          # Agent 2 ← receives Agent 1's output
    code         = run_code_generator(design)           # Agent 3 ← receives Agent 2's output
    tests        = run_qa_writer(code)                  # Agent 4 ← receives Agent 3's output
    return requirements, design, code, tests

In [ ]:
# ── Gradio UI — Handoff Pipeline ─────────────────────────────────────────────
with gr.Blocks(title="A2A: Handoff Pipeline") as handoff_demo:
    gr.Markdown(
        "## Pattern 1 — Handoff: Feature Request Pipeline\n"
        "Four specialist agents hand off in sequence: "
        "**Requirements Analyst → Software Architect → Code Generator → QA Test Writer**\n\n"
        "Each agent receives only the _previous agent's output_, not the original request."
    )
    with gr.Row():
        with gr.Column(scale=1):
            handoff_input = gr.Textbox(
                label="Raw Feature Request",
                placeholder="e.g. Add rate limiting to our REST API so free-tier users can make at most 100 requests per hour",
                lines=4
            )
            handoff_btn = gr.Button("Run Pipeline", variant="primary")
        with gr.Column(scale=2):
            with gr.Row():
                h_req = gr.Textbox(label="Agent 1 — Requirements Analyst", lines=12, interactive=False)
                h_des = gr.Textbox(label="Agent 2 — Software Architect",   lines=12, interactive=False)
            with gr.Row():
                h_code = gr.Code(label="Agent 3 — Code Generator", language="python")
                h_test = gr.Code(label="Agent 4 — QA Test Writer",  language="python")

    handoff_btn.click(
        fn=run_handoff_pipeline,
        inputs=handoff_input,
        outputs=[h_req, h_des, h_code, h_test]
    )
    gr.Examples(
        examples=[
            ["Add rate limiting to our REST API — free tier: 100 requests/hour, paid tier: 10,000 requests/hour. Enforce per API key."],
            ["Build a webhook delivery system that retries failed deliveries with exponential backoff up to 5 attempts."],
            ["Implement a feature flag service so we can enable/disable features per user cohort without a deployment."],
            ["Add audit logging to all database write operations — record who changed what, when, and the before/after values."],
        ],
        inputs=handoff_input
    )

handoff_demo.launch()

---
# Pattern 2 — Supervisor / Orchestrator

## Concept

A **Supervisor** agent sits at the centre. It receives the user goal, produces a
dispatch plan, sends sub-tasks to specialist **Worker** agents (running in **parallel**),
then synthesises all results into the final deliverable.

```
                    ┌──► [Worker: Security Reviewer]       ──┐
                    │                                         │
Code ──► [Supervisor plans] ──► [Worker: Performance Reviewer] ──► [Supervisor synthesises] ──► Report
                    │                                         │
                    └──► [Worker: Maintainability Reviewer]  ──┘
                              ↑ all three run in parallel
```

## A2A Mechanics
- Supervisor produces a **JSON dispatch plan** listing which workers to call and what to focus on
- Workers run in **parallel** via `ThreadPoolExecutor` — no worker waits for another
- Supervisor receives all worker outputs and performs a **synthesis pass** (its own LLM call)

## Difference from Orchestrator-Workers (Agents.ipynb Pattern 4)

| Agents.ipynb Pattern 4 | This Pattern |
|---|---|
| Orchestrator plans *research sub-topics* | Supervisor plans *review dimensions* |
| Workers produce independent research blobs | Workers produce structured findings per category |
| Output is a narrative report | Output is a prioritised action plan |

## Agents

| Agent | Role | Input | Output |
|---|---|---|---|
| Supervisor (plan) | Decide what each reviewer focuses on | Code + meta-instruction | JSON dispatch plan |
| Security Reviewer | OWASP, injection, auth, secrets | Code + focus instruction | Numbered findings |
| Performance Reviewer | Complexity, N+1, caching | Code + focus instruction | Numbered findings |
| Maintainability Reviewer | Naming, coupling, test gaps | Code + focus instruction | Numbered findings |
| Supervisor (synthesise) | Merge and prioritise all findings | 3 review reports | Prioritised action plan |

In [ ]:
# ── Pattern 2: Supervisor / Orchestrator — Code Review ───────────────────────

SUPERVISOR_PLAN_SYSTEM = """You are a Lead Engineering Manager running a code review.
Dispatch three specialist reviewers by returning ONLY a valid JSON array of exactly 3 objects:
[
  {"reviewer": "Security",        "focus": "<specific instruction tailored to this code>"},
  {"reviewer": "Performance",     "focus": "<specific instruction tailored to this code>"},
  {"reviewer": "Maintainability", "focus": "<specific instruction tailored to this code>"}
]
Each focus instruction must reference something specific you see in the submitted code.
Output ONLY the JSON array — no other text."""

REVIEWER_SYSTEMS = {
    "Security": (
        "You are a Security Specialist performing a code review. "
        "Focus on OWASP Top-10 vulnerabilities, SQL/command injection, hardcoded secrets, "
        "improper authentication, insecure deserialization, and insufficient input validation. "
        "Format findings as a numbered list. Each item: [CRITICAL/HIGH/MEDIUM/LOW] — location — description — recommended fix."
    ),
    "Performance": (
        "You are a Performance Engineer performing a code review. "
        "Focus on algorithmic complexity, N+1 query patterns, unnecessary I/O, "
        "memory leaks, missing database indexes, and caching opportunities. "
        "Format findings as a numbered list. Each item: [HIGH/MEDIUM/LOW] impact — location — description — recommended optimisation."
    ),
    "Maintainability": (
        "You are a Code Quality Specialist performing a code review. "
        "Focus on naming clarity, function length, cyclomatic complexity, tight coupling, "
        "missing docstrings, insufficient error handling, and test coverage gaps. "
        "Format findings as a numbered list. Each item: [HIGH/MEDIUM/LOW] — location — description — recommended refactor."
    ),
}

SUPERVISOR_SYNTH_SYSTEM = """You are a Lead Engineering Manager synthesising code review results.
You have received reports from three specialist reviewers: Security, Performance, and Maintainability.

Produce a **Prioritised Action Plan**:
1. Merge all findings and sort by priority: CRITICAL first, then HIGH, MEDIUM, LOW
2. Format each item: [Priority] **Area** — Finding — Recommended Action
3. End with a **Summary** section:
   - Overall code quality score /10
   - Top 3 immediate actions before merging
   - One-sentence merge readiness verdict

Be direct and actionable."""


def dispatch_worker(reviewer: str, focus: str, code: str) -> tuple[str, str]:
    system = REVIEWER_SYSTEMS.get(reviewer, REVIEWER_SYSTEMS["Maintainability"])
    result = llm_call(
        system,
        f"Focus instruction: {focus}\n\nCode to review:\n```python\n{code}\n```"
    )
    return reviewer, result


def run_supervisor_review(code: str) -> tuple[str, str, str, str, str]:
    """Supervisor plans dispatch, workers run in parallel, supervisor synthesises."""
    # ── Step 1: Supervisor plans ──────────────────────────────────────────────
    plan_raw = llm_call(
        SUPERVISOR_PLAN_SYSTEM,
        f"Code submitted for review:\n```python\n{code}\n```"
    )
    try:
        text = plan_raw.strip()
        if "```" in text:
            text = text.split("```")[1].lstrip("json").strip()
        dispatch_plan = json.loads(text)
    except Exception:
        dispatch_plan = [
            {"reviewer": "Security",        "focus": "General security review"},
            {"reviewer": "Performance",     "focus": "General performance review"},
            {"reviewer": "Maintainability", "focus": "General code quality review"},
        ]

    plan_display = "**Supervisor Dispatch Plan:**\n" + "\n".join(
        f"- **{item['reviewer']}**: {item['focus']}" for item in dispatch_plan
    )

    # ── Step 2: Workers run in parallel ──────────────────────────────────────
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [
            executor.submit(dispatch_worker, item["reviewer"], item["focus"], code)
            for item in dispatch_plan
        ]
        worker_results = dict(f.result() for f in concurrent.futures.as_completed(futures))

    sec_result   = worker_results.get("Security",        "No findings.")
    perf_result  = worker_results.get("Performance",     "No findings.")
    maint_result = worker_results.get("Maintainability", "No findings.")

    # ── Step 3: Supervisor synthesises ───────────────────────────────────────
    synthesis_input = (
        f"**Security Review:**\n{sec_result}\n\n"
        f"**Performance Review:**\n{perf_result}\n\n"
        f"**Maintainability Review:**\n{maint_result}"
    )
    action_plan = llm_call(SUPERVISOR_SYNTH_SYSTEM, synthesis_input)

    return plan_display, sec_result, perf_result, maint_result, action_plan

In [ ]:
# ── Gradio UI — Supervisor Code Review ───────────────────────────────────────
_REVIEW_EXAMPLE_1 = """import sqlite3

def get_user(username: str):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query = f"SELECT * FROM users WHERE username = '{username}'"
    cursor.execute(query)
    results = []
    for row in cursor.fetchall():
        results.append(row)
    conn.close()
    return results

def process_all_users():
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute('SELECT id FROM users')
    user_ids = cursor.fetchall()
    output = []
    for uid in user_ids:
        cursor.execute(f'SELECT * FROM orders WHERE user_id = {uid[0]}')
        orders = cursor.fetchall()
        for order in orders:
            cursor.execute(f'SELECT * FROM items WHERE order_id = {order[0]}')
            items = cursor.fetchall()
            output.append({'user': uid, 'order': order, 'items': items})
    conn.close()
    return output
"""

_REVIEW_EXAMPLE_2 = """import hashlib
import os

SECRET_KEY = 'mysecretkey123'
DB_PASSWORD = 'admin123'

class AuthService:
    def __init__(self):
        self.users = {}
        self.sessions = {}

    def register(self, username, password, email, phone, address, dob, ssn):
        hashed = hashlib.md5(password.encode()).hexdigest()
        self.users[username] = {
            'password': hashed, 'email': email, 'phone': phone,
            'address': address, 'dob': dob, 'ssn': ssn
        }
        return True

    def login(self, username, password):
        if username in self.users:
            hashed = hashlib.md5(password.encode()).hexdigest()
            if self.users[username]['password'] == hashed:
                token = os.urandom(8).hex()
                self.sessions[token] = username
                return token
        return None

    def get_user_data(self, token, requested_username):
        if token in self.sessions:
            return self.users.get(requested_username)
        return None
"""

with gr.Blocks(title="A2A: Supervisor Code Review") as supervisor_demo:
    gr.Markdown(
        "## Pattern 2 — Supervisor / Orchestrator: Multi-Dimension Code Review\n"
        "The **Supervisor** dispatches three specialist workers in parallel "
        "(Security, Performance, Maintainability), then synthesises their findings "
        "into a prioritised action plan."
    )
    with gr.Row():
        with gr.Column(scale=1):
            sup_input = gr.Code(
                label="Python Code to Review",
                language="python",
                value=_REVIEW_EXAMPLE_1
            )
            sup_btn = gr.Button("Run Review", variant="primary")
        with gr.Column(scale=2):
            sup_plan  = gr.Markdown(label="Supervisor Dispatch Plan")
            with gr.Row():
                sup_sec   = gr.Textbox(label="Security Findings",        lines=10, interactive=False)
                sup_perf  = gr.Textbox(label="Performance Findings",     lines=10, interactive=False)
                sup_maint = gr.Textbox(label="Maintainability Findings", lines=10, interactive=False)
            sup_action = gr.Markdown(label="Prioritised Action Plan")

    sup_btn.click(
        fn=run_supervisor_review,
        inputs=sup_input,
        outputs=[sup_plan, sup_sec, sup_perf, sup_maint, sup_action]
    )
    gr.Examples(
        examples=[[_REVIEW_EXAMPLE_1], [_REVIEW_EXAMPLE_2]],
        inputs=sup_input
    )

supervisor_demo.launch()

---
# Pattern 3 — Peer Debate / Collaboration

## Concept

Two agents with **different design philosophies** each propose a solution to the same problem.
They then critique each other's proposal, issue rebuttals, and a **Judge agent** reads all
four documents and picks the stronger approach with explicit reasoning.

```
Problem
  │
  ├──► [Agent A: Pragmatist]  ── proposes solution A ──► critiques B's proposal
  │         │                                                    │
  │         │  (runs in parallel)               (runs in parallel)
  │         │                                                    │
  └──► [Agent B: Purist]      ── proposes solution B ──► critiques A's proposal
              │                                                    │
              └── rebuts critique of B  (parallel) ───────────────┘
              │                         Agent A rebuts critique of A
              ▼
        [Judge Agent]
              │  reads all 4 documents
              ▼
        Final Verdict
```

## A2A Mechanics
- **Round 0** (parallel): Both agents propose independently
- **Round 1** (parallel): Agent A critiques B's proposal; Agent B critiques A's proposal
- **Round 2** (parallel): Each agent rebuts the critique of their own proposal
- **Judgment**: Judge reads all 4 documents and decides

## When to Use
- Architectural decisions where multiple valid approaches exist
- When you want to surface trade-offs and assumptions, not just get an answer
- When the cost of a wrong decision is high

## Agents

| Agent | Persona | Task |
|---|---|---|
| Proposer A (Pragmatist) | Values simplicity and delivery speed | Proposes a practical, proven solution |
| Proposer B (Purist) | Values correctness and long-term design | Proposes a principled, scalable solution |
| Critiquer (cross) | Finds weaknesses in the other agent's proposal | Returns 3 specific criticisms |
| Rebutter (self-defence) | Defends own proposal against critique | Addresses each criticism |
| Judge | Neutral arbiter | Picks the stronger solution with structured reasoning |

In [ ]:
# ── Pattern 3: Peer Debate — Architecture Decision ───────────────────────────

PROPOSER_A_SYSTEM = """You are a Pragmatic Software Architect.
You value simplicity, fast delivery, and proven technology over theoretical purity.
Given an architecture problem, propose a practical solution that a small team can ship quickly.
Structure your proposal:
## Proposal (Pragmatist Approach)
**Chosen Approach:** <name>
**Rationale:** <2-3 sentences>
**Implementation Sketch:** <key components and how they interact>
**Trade-offs accepted:** <what you are intentionally giving up for simplicity>"""

PROPOSER_B_SYSTEM = """You are a Principled Software Architect.
You value correctness, long-term maintainability, and clean separation of concerns.
Given an architecture problem, propose the most robust, scalable solution even if it takes longer to build.
Structure your proposal:
## Proposal (Principled Approach)
**Chosen Approach:** <name>
**Rationale:** <2-3 sentences>
**Implementation Sketch:** <key components and how they interact>
**Trade-offs accepted:** <what you are intentionally giving up for correctness>"""

CRITIQUER_SYSTEM = """You are a rigorous Technical Reviewer.
You have received a proposal from another architect. Find its weaknesses.
Produce exactly 3 criticisms, each with:
**Criticism {n}:** <title>
- **Issue:** <specific problem with the proposal>
- **Risk:** <what goes wrong if this issue is not addressed>
- **Evidence:** <reference to a specific claim or component in the proposal>
Be specific — vague criticisms are not useful."""

REBUTTER_SYSTEM = """You are defending your architecture proposal against a critique.
Address each criticism directly and explain why your proposal still stands or how you would amend it.
Format:
## Rebuttal
**Re: Criticism 1 — <title>:** <response>
**Re: Criticism 2 — <title>:** <response>
**Re: Criticism 3 — <title>:** <response>
**Overall position:** <one sentence summary of why your approach is the right choice>"""

JUDGE_SYSTEM = """You are a Principal Engineer and neutral arbiter of a technical debate.
You have read two competing proposals, the cross-critiques, and the rebuttals.
Produce a structured verdict:
## Verdict
**Winner:** <Pragmatist Approach OR Principled Approach>
**Decisive Factor:** <the single most important reason for your decision>
**Strengths of the winner:** <2 bullet points>
**Valid concerns from the loser:** <1-2 points the losing team raised that should still be addressed>
**Recommendation:** <one concrete next step for the team>
Base your decision on the problem statement, the proposals, and the debate — not on general preferences."""


def run_debate(problem_statement: str) -> tuple[str, str, str, str, str, str, str]:
    """Run the full two-agent debate and return all artefacts."""
    # ── Round 0: Both agents propose (parallel) ───────────────────────────────
    with concurrent.futures.ThreadPoolExecutor() as ex:
        fut_a = ex.submit(llm_call, PROPOSER_A_SYSTEM, f"Architecture problem:\n{problem_statement}")
        fut_b = ex.submit(llm_call, PROPOSER_B_SYSTEM, f"Architecture problem:\n{problem_statement}")
        proposal_a = fut_a.result()
        proposal_b = fut_b.result()

    # ── Round 1: Cross-critique (parallel) ───────────────────────────────────
    with concurrent.futures.ThreadPoolExecutor() as ex:
        # A critiques B's proposal; B critiques A's proposal
        fut_crit_a = ex.submit(
            llm_call, CRITIQUER_SYSTEM,
            f"Problem statement:\n{problem_statement}\n\nProposal to critique:\n{proposal_b}"
        )
        fut_crit_b = ex.submit(
            llm_call, CRITIQUER_SYSTEM,
            f"Problem statement:\n{problem_statement}\n\nProposal to critique:\n{proposal_a}"
        )
        critique_a_of_b = fut_crit_a.result()   # A's critique of B
        critique_b_of_a = fut_crit_b.result()   # B's critique of A

    # ── Round 2: Rebuttals (parallel) ────────────────────────────────────────
    with concurrent.futures.ThreadPoolExecutor() as ex:
        fut_reb_a = ex.submit(
            llm_call, REBUTTER_SYSTEM,
            f"Your proposal:\n{proposal_a}\n\nCritique you received:\n{critique_b_of_a}"
        )
        fut_reb_b = ex.submit(
            llm_call, REBUTTER_SYSTEM,
            f"Your proposal:\n{proposal_b}\n\nCritique you received:\n{critique_a_of_b}"
        )
        rebuttal_a = fut_reb_a.result()
        rebuttal_b = fut_reb_b.result()

    # ── Judgment ──────────────────────────────────────────────────────────────
    judge_input = (
        f"**Problem Statement:**\n{problem_statement}\n\n"
        f"**Pragmatist Proposal:**\n{proposal_a}\n\n"
        f"**Principled Proposal:**\n{proposal_b}\n\n"
        f"**Critique of Principled (by Pragmatist):**\n{critique_a_of_b}\n\n"
        f"**Critique of Pragmatist (by Principled):**\n{critique_b_of_a}\n\n"
        f"**Pragmatist Rebuttal:**\n{rebuttal_a}\n\n"
        f"**Principled Rebuttal:**\n{rebuttal_b}"
    )
    verdict = llm_call(JUDGE_SYSTEM, judge_input)

    return proposal_a, proposal_b, critique_a_of_b, critique_b_of_a, rebuttal_a, rebuttal_b, verdict

In [ ]:
# ── Gradio UI — Peer Debate ───────────────────────────────────────────────────
with gr.Blocks(title="A2A: Peer Debate") as debate_demo:
    gr.Markdown(
        "## Pattern 3 — Peer Debate / Collaboration: Architecture Decision\n"
        "Two architects (Pragmatist vs Principled) propose solutions, "
        "critique each other, issue rebuttals, and a **Judge** picks the winner.\n\n"
        "All proposal + critique rounds run in **parallel**."
    )
    with gr.Row():
        with gr.Column(scale=1):
            debate_input = gr.Textbox(
                label="Architecture Problem",
                placeholder="e.g. How should we handle async job processing for a SaaS app expecting 10,000 jobs/day?",
                lines=4
            )
            debate_btn = gr.Button("Start Debate", variant="primary")
        with gr.Column(scale=3):
            with gr.Row():
                d_prop_a = gr.Textbox(label="Proposal A — Pragmatist",  lines=10, interactive=False)
                d_prop_b = gr.Textbox(label="Proposal B — Principled",  lines=10, interactive=False)
            with gr.Row():
                d_crit_a = gr.Textbox(label="A critiques B", lines=8, interactive=False)
                d_crit_b = gr.Textbox(label="B critiques A", lines=8, interactive=False)
            with gr.Row():
                d_reb_a  = gr.Textbox(label="A rebuts B's critique", lines=8, interactive=False)
                d_reb_b  = gr.Textbox(label="B rebuts A's critique", lines=8, interactive=False)
            d_verdict = gr.Markdown(label="Judge's Verdict")

    debate_btn.click(
        fn=run_debate,
        inputs=debate_input,
        outputs=[d_prop_a, d_prop_b, d_crit_a, d_crit_b, d_reb_a, d_reb_b, d_verdict]
    )
    gr.Examples(
        examples=[
            ["How should we implement async job processing for a SaaS platform expecting 10,000 jobs/day? We have a small team of 4 engineers."],
            ["Should we split our monolithic Django app into microservices? We have 50k daily active users and 8 engineers."],
            ["How should we implement the data layer for a new analytics dashboard — relational DB or a document store?"],
        ],
        inputs=debate_input
    )

debate_demo.launch()

---
# Pattern 4 — Critic / Reflection

## Concept

A **Generator** agent produces code. A **Critic** agent evaluates it with a score
and structured feedback (strengths + improvements). The Generator receives the
feedback and produces an improved version. The loop continues until the Critic's
score reaches 8/10 or max iterations are exhausted.

```
Task + Requirements
        │
        ▼
  [Generator]  ◄─────────────── feedback (improvements)
        │                              │
        │  code                        │
        ▼                              │
    [Critic]  ── score >= 8? ──YES──► Accept
        │
       NO ──► loop back to Generator
```

## Difference from Evaluator-Optimizer (Agents.ipynb Pattern 5)

| Agents.ipynb Pattern 5 | This Pattern |
|---|---|
| Evaluator returns `{score, passed, feedback}` | Critic returns `{score, passed, strengths, improvements}` |
| Feedback is a single string | Feedback distinguishes *what works* from *what to fix* |
| Trace shows pass/fail per iteration | Trace shows score progression + strength summary |
| Generator only sees what to improve | Generator sees strengths to preserve AND improvements to make |

## When to Use
- Any task where quality is measurable and iteration is cheap (code, writing, design)
- When you want the agent to learn from its own mistakes within a single run
- When the first attempt is rarely good enough

## Agents

| Agent | Role | Input | Output |
|---|---|---|---|
| Generator | Write production Python code | Task + requirements + (feedback from previous iteration) | Python code |
| Critic | Score and analyse the code | Code + requirements | JSON: `{score, passed, strengths, improvements}` |

In [ ]:
# ── Pattern 4: Critic / Reflection — Iterative Code Improvement ──────────────

GENERATOR_SYSTEM = """You are a Senior Python Developer writing production-quality code.
- Use clear type hints on all functions
- Write docstrings for all public functions and classes
- Handle edge cases explicitly
- Use stdlib unless a specific library is required
If you received feedback from a previous review, address every point.
Preserve what the critic said was already good.
Output ONLY the Python code — no explanation before or after."""

CRITIC_SYSTEM = """You are a strict Senior Code Reviewer at a top tech company.
Evaluate the code against the given requirements and reply ONLY in this exact JSON format:
{
  "score": <integer 1-10>,
  "passed": <true if score >= 8, else false>,
  "strengths": ["<what is already good — be specific>", "<strength 2>"],
  "improvements": ["<specific change needed>", "<specific change needed>", ...]
}
Score guide: 1-4 = major issues, 5-7 = mostly correct but needs work, 8-9 = good, 10 = exceptional.
Be strict: a score of 8 means the code is genuinely production-ready.
Output ONLY the JSON — no other text."""


def generate_code(task: str, requirements: str, feedback: dict | None) -> str:
    prompt = f"Task: {task}\n\nRequirements:\n{requirements}"
    if feedback:
        strengths    = "\n".join(f"- {s}" for s in feedback.get("strengths", []))
        improvements = "\n".join(f"- {i}" for i in feedback.get("improvements", []))
        prompt += (
            f"\n\nCritic's feedback from previous iteration:"
            f"\n**Keep these strengths:**\n{strengths}"
            f"\n**Address these improvements:**\n{improvements}"
        )
    return llm_call(GENERATOR_SYSTEM, prompt)


def critique_code(code: str, requirements: str) -> dict:
    raw = llm_call(
        CRITIC_SYSTEM,
        f"Requirements:\n{requirements}\n\nCode to review:\n```python\n{code}\n```"
    )
    try:
        text = raw.strip()
        if "```" in text:
            text = text.split("```")[1].lstrip("json").strip()
        return json.loads(text)
    except Exception:
        return {"score": 5, "passed": False, "strengths": [], "improvements": [raw]}


def run_reflection_loop(
    task: str, requirements: str, max_iter: int = 4
) -> tuple[str, str, str]:
    """Generator-Critic loop. Returns (trace, final_code, final_critique_markdown)."""
    trace_lines: list[str] = []
    feedback    = None
    final_code  = ""
    final_eval  = {}

    for i in range(1, max_iter + 1):
        # ── Generator writes / rewrites ───────────────────────────────────────
        code     = generate_code(task, requirements, feedback)
        final_code = code

        # ── Critic evaluates ──────────────────────────────────────────────────
        evaluation = critique_code(code, requirements)
        final_eval = evaluation
        score      = evaluation.get("score", 0)
        passed     = evaluation.get("passed", False)
        strengths  = evaluation.get("strengths", [])
        improvements = evaluation.get("improvements", [])

        status = "PASSED" if passed else "needs improvement"
        trace_lines.append(f"**Iteration {i}:** Score `{score}/10` — {status}")
        if strengths:
            trace_lines.append("  - Strengths: " + " | ".join(strengths[:2]))
        if improvements and not passed:
            trace_lines.append("  - Improvements: " + " | ".join(improvements[:2]))

        if passed:
            trace_lines.append(f"\nAccepted after **{i}** iteration(s).")
            break
        feedback = evaluation
    else:
        trace_lines.append("\nMax iterations reached — using final version.")

    # ── Format final critique as markdown ─────────────────────────────────────
    strengths_md    = "\n".join(f"- {s}" for s in final_eval.get("strengths", []))
    improvements_md = "\n".join(f"- {i}" for i in final_eval.get("improvements", []))
    final_critique = (
        f"**Final Score: {final_eval.get('score', '?')}/10**\n\n"
        f"**Strengths:**\n{strengths_md or 'None noted.'}\n\n"
        f"**Remaining improvements:**\n{improvements_md or 'None — code passed.'}"
    )

    return "\n".join(trace_lines), final_code, final_critique

In [ ]:
# ── Gradio UI — Critic / Reflection ──────────────────────────────────────────
with gr.Blocks(title="A2A: Critic / Reflection") as reflection_demo:
    gr.Markdown(
        "## Pattern 4 — Critic / Reflection: Iterative Code Improvement\n"
        "A **Generator** writes code; a **Critic** scores it (1–10) and provides "
        "strengths + improvements; the Generator uses both to produce a better version.\n\n"
        "Loop continues until score ≥ 8 or max iterations reached."
    )
    with gr.Row():
        with gr.Column():
            ref_task  = gr.Textbox(
                label="Coding Task", lines=2,
                placeholder="e.g. Implement an LRU cache with get and put operations"
            )
            ref_reqs  = gr.Textbox(
                label="Requirements / Acceptance Criteria", lines=5,
                placeholder="- Signature: ...\n- Edge cases: ..."
            )
            ref_iter  = gr.Slider(label="Max Iterations", minimum=1, maximum=5, step=1, value=3)
            ref_btn   = gr.Button("Run Reflection Loop", variant="primary")
        with gr.Column():
            ref_trace    = gr.Markdown(label="Iteration Trace (score progression)")
            ref_code     = gr.Code(label="Final Generated Code", language="python")
            ref_critique = gr.Markdown(label="Critic's Final Assessment")

    ref_btn.click(
        fn=run_reflection_loop,
        inputs=[ref_task, ref_reqs, ref_iter],
        outputs=[ref_trace, ref_code, ref_critique]
    )
    gr.Examples(
        examples=[
            [
                "Implement an LRU (Least Recently Used) cache",
                "- Class: LRUCache(capacity: int)\n- Methods: get(key: int) -> int (-1 if missing), put(key: int, value: int) -> None\n- O(1) average time for both operations\n- Evicts least recently used item when capacity exceeded\n- Thread-safe implementation",
                3
            ],
            [
                "Implement an async task queue with worker pool",
                "- Class: TaskQueue(max_workers: int)\n- Methods: submit(coro) to enqueue, start() to begin processing, stop() to drain and shut down\n- Uses asyncio\n- Workers run concurrently up to max_workers limit\n- Graceful shutdown: completes in-progress tasks before stopping",
                3
            ],
        ],
        inputs=[ref_task, ref_reqs, ref_iter]
    )

reflection_demo.launch()

---
# Pattern 5 — Agent as a Tool

## Concept

An **Orchestrator** agent calls other agents **as if they were tools** — registered in the
tool definitions just like any function. Each sub-agent has its own LLM, system prompt,
and internal tools. From the orchestrator's perspective, calling a sub-agent looks
identical to calling `get_weather()` or `query_database()`.

```
User Goal
     │
     ▼
[Orchestrator Agent]
     │
     ├── tool_call: "research_agent(topic)"  ──► [Research Agent]
     │                                               └── uses: web_search, summarise
     │
     ├── tool_call: "coding_agent(spec)"     ──► [Coding Agent]
     │                                               └── uses: own LLM + code tools
     │
     └── tool_call: "qa_agent(code)"         ──► [QA Agent]
                                                     └── uses: own LLM + test tools
```

## Why This Is Different from the Other Patterns

| Other Patterns | Agent as a Tool |
|---|---|
| Orchestrator knows about each agent explicitly | Orchestrator only knows tool names + descriptions |
| Agent communication is hardcoded in pipeline | Orchestrator **decides at runtime** which sub-agents to call |
| Sub-agents are just `llm_call` wrappers | Sub-agents are **full agent loops** with their own tools |
| Fixed flow | Dynamic flow — orchestrator may call the same sub-agent multiple times |

The key insight: **encapsulation**. The orchestrator has no idea that `coding_agent` internally
runs its own loop with `run_tests` and `lint_code` tools. It just gets a result back.

## When to Use
- Large systems where sub-problems are complex enough to need their own agent loops
- When sub-agents should be reusable across different orchestrators
- When you want dynamic dispatch — the orchestrator decides which sub-agents are needed based on the input
- Foundation for production multi-agent platforms (OpenAI Swarm, Claude Agent SDK, AutoGen)

## Scenario: Software Delivery Orchestrator

A project manager describes a feature. The Orchestrator decides which sub-agents to invoke
and in what order:

| Sub-Agent Tool | Internal Loop | Output |
|---|---|---|
| `research_agent` | Surveys best practices + libraries for the feature | Research brief |
| `coding_agent` | Writes implementation based on spec + research | Python code |
| `qa_agent` | Reviews the code and writes a test plan | QA report |

The Orchestrator dynamically decides whether to call all three, skip research for simple tasks,
or call `coding_agent` twice if the first result is insufficient.

In [ ]:
# ── Pattern 5: Agent as a Tool — Software Delivery Orchestrator ──────────────

# ── Sub-Agent 1: Research Agent ───────────────────────────────────────────────
RESEARCH_AGENT_SYSTEM = """You are a Senior Technical Researcher.
Given a software feature specification, survey best practices, common patterns,
and recommended libraries for implementing it in Python.

Structure your output:
## Research Brief: <feature name>
**Recommended Approach:** <chosen pattern / architecture>
**Key Libraries:** <relevant stdlib or third-party packages with one-line descriptions>
**Best Practices:** <3-4 bullet points specific to this type of feature>
**Pitfalls to Avoid:** <2-3 common mistakes>
**Reference Implementations:** <brief description of how similar systems solve this>

Be specific and actionable — the developer will use this to write the implementation."""

# Research agent's internal tools (simulated as functions)
def _search_docs(query: str) -> str:
    """Simulate a documentation search — in production this would hit a real search API."""
    return f"[Docs search result for '{query}': Found relevant patterns in Python stdlib and common OSS libraries]"

def _check_libraries(feature_type: str) -> str:
    """Simulate checking available libraries for a feature type."""
    return f"[Library check for '{feature_type}': Found multiple mature options with active maintenance]"

RESEARCH_INTERNAL_TOOLS = [
    {"type": "function", "function": {
        "name": "_search_docs",
        "description": "Search technical documentation for patterns related to a query",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "_check_libraries",
        "description": "Check available Python libraries for a given feature type",
        "parameters": {"type": "object", "properties": {
            "feature_type": {"type": "string"}}, "required": ["feature_type"], "additionalProperties": False}}},
]
RESEARCH_TOOL_MAP = {"_search_docs": _search_docs, "_check_libraries": _check_libraries}

def run_research_agent(feature_spec: str) -> str:
    """Research sub-agent with its own internal tool loop."""
    messages = [
        {"role": "system", "content": RESEARCH_AGENT_SYSTEM},
        {"role": "user",   "content": f"Research this feature:\n{feature_spec}"},
    ]
    for _ in range(5):
        resp   = client.chat.completions.create(model=MODEL, messages=messages, tools=RESEARCH_INTERNAL_TOOLS)
        msg    = resp.choices[0].message
        finish = resp.choices[0].finish_reason
        if finish == "stop":
            return msg.content
        if finish == "tool_calls":
            messages.append(msg)
            for tc in msg.tool_calls:
                result = RESEARCH_TOOL_MAP[tc.function.name](**json.loads(tc.function.arguments))
                messages.append({"role": "tool", "content": result, "tool_call_id": tc.id})
    return messages[-1]["content"] if messages else "Research incomplete."


# ── Sub-Agent 2: Coding Agent ─────────────────────────────────────────────────
CODING_AGENT_SYSTEM = """You are a Senior Python Developer.
You receive a feature specification and optionally a research brief.
Write clean, production-quality Python:
- Type hints on all functions and classes
- Docstrings on all public interfaces
- Explicit edge case handling
- Use only stdlib unless a library is explicitly mentioned in the research brief
Output ONLY the Python code — no preamble or explanation."""

def _lint_check(code: str) -> str:
    """Simulate a lint check — counts obvious issues."""
    issues = []
    if "except:" in code:        issues.append("bare except clause")
    if "print(" in code:         issues.append("debug print statement found")
    if "TODO" in code:           issues.append("unresolved TODO comment")
    return f"Lint result: {', '.join(issues) if issues else 'no issues found'}"

CODING_INTERNAL_TOOLS = [
    {"type": "function", "function": {
        "name": "_lint_check",
        "description": "Run a lint check on Python code and return any issues found",
        "parameters": {"type": "object", "properties": {
            "code": {"type": "string"}}, "required": ["code"], "additionalProperties": False}}},
]
CODING_TOOL_MAP = {"_lint_check": _lint_check}

def run_coding_agent(spec: str, research_brief: str = "") -> str:
    """Coding sub-agent with its own lint-check tool loop."""
    user_msg = f"Feature spec:\n{spec}"
    if research_brief:
        user_msg += f"\n\nResearch brief to follow:\n{research_brief}"
    messages = [
        {"role": "system", "content": CODING_AGENT_SYSTEM},
        {"role": "user",   "content": user_msg},
    ]
    for _ in range(6):
        resp   = client.chat.completions.create(model=MODEL, messages=messages, tools=CODING_INTERNAL_TOOLS)
        msg    = resp.choices[0].message
        finish = resp.choices[0].finish_reason
        if finish == "stop":
            return msg.content
        if finish == "tool_calls":
            messages.append(msg)
            for tc in msg.tool_calls:
                result = CODING_TOOL_MAP[tc.function.name](**json.loads(tc.function.arguments))
                messages.append({"role": "tool", "content": result, "tool_call_id": tc.id})
    return messages[-1]["content"] if messages else "Coding incomplete."


# ── Sub-Agent 3: QA Agent ─────────────────────────────────────────────────────
QA_AGENT_SYSTEM = """You are a Senior QA Engineer.
You receive implementation code and produce a QA report covering:
## QA Report
**Code Quality Score:** <n>/10
**Test Coverage Plan:**
- <test case>: <what it validates>
**Critical Risks:** <top issues that must be fixed before shipping>
**Suggested pytest snippet:** <one concrete test function demonstrating the most important test>
Be specific — reference actual function names and edge cases from the code."""

def _run_static_analysis(code: str) -> str:
    """Simulate static analysis."""
    lines   = code.count("\n")
    funcs   = code.count("def ")
    classes = code.count("class ")
    return f"Static analysis: {lines} lines, {funcs} functions, {classes} classes. Complexity: {'high' if funcs > 8 else 'moderate' if funcs > 4 else 'low'}."

QA_INTERNAL_TOOLS = [
    {"type": "function", "function": {
        "name": "_run_static_analysis",
        "description": "Run static analysis on Python code to measure size and complexity",
        "parameters": {"type": "object", "properties": {
            "code": {"type": "string"}}, "required": ["code"], "additionalProperties": False}}},
]
QA_TOOL_MAP = {"_run_static_analysis": _run_static_analysis}

def run_qa_agent(code: str) -> str:
    """QA sub-agent with its own static analysis tool loop."""
    messages = [
        {"role": "system", "content": QA_AGENT_SYSTEM},
        {"role": "user",   "content": f"Review this code:\n```python\n{code}\n```"},
    ]
    for _ in range(5):
        resp   = client.chat.completions.create(model=MODEL, messages=messages, tools=QA_INTERNAL_TOOLS)
        msg    = resp.choices[0].message
        finish = resp.choices[0].finish_reason
        if finish == "stop":
            return msg.content
        if finish == "tool_calls":
            messages.append(msg)
            for tc in msg.tool_calls:
                result = QA_TOOL_MAP[tc.function.name](**json.loads(tc.function.arguments))
                messages.append({"role": "tool", "content": result, "tool_call_id": tc.id})
    return messages[-1]["content"] if messages else "QA incomplete."


# ── Orchestrator — registers sub-agents as tools ──────────────────────────────
ORCHESTRATOR_SYSTEM = """You are a Software Delivery Orchestrator.
You have three specialist sub-agents available as tools:
- research_agent: surveys best practices and libraries for a feature
- coding_agent:   implements the feature (optionally using research output)
- qa_agent:       reviews the implementation and produces a QA report

Given a feature request, decide which agents to call and in what order.
For simple features you may skip research. For complex ones, always research first.
Pass the research output into coding_agent when available.
Always run qa_agent on the final code.
When all agents have reported, summarise the delivery artefacts in a short handover note."""

ORCHESTRATOR_TOOLS = [
    {"type": "function", "function": {
        "name": "research_agent",
        "description": "Survey best practices and recommend libraries for a software feature. Call this first for non-trivial features.",
        "parameters": {"type": "object", "properties": {
            "feature_spec": {"type": "string", "description": "Description of the feature to research"}},
            "required": ["feature_spec"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "coding_agent",
        "description": "Implement a software feature in Python. Optionally pass a research_brief from research_agent.",
        "parameters": {"type": "object", "properties": {
            "spec":           {"type": "string", "description": "Feature specification to implement"},
            "research_brief": {"type": "string", "description": "Optional research output to guide implementation", "default": ""}},
            "required": ["spec"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "qa_agent",
        "description": "Review Python implementation code and produce a QA report with test coverage plan.",
        "parameters": {"type": "object", "properties": {
            "code": {"type": "string", "description": "Python code to review"}},
            "required": ["code"], "additionalProperties": False}}},
]

SUB_AGENT_MAP = {
    "research_agent": lambda args: run_research_agent(args["feature_spec"]),
    "coding_agent":   lambda args: run_coding_agent(args["spec"], args.get("research_brief", "")),
    "qa_agent":       lambda args: run_qa_agent(args["code"]),
}


def run_orchestrator_agent(feature_request: str) -> tuple[str, str]:
    """
    Orchestrator agent loop — dispatches sub-agents as tool calls.
    Returns (trace_markdown, final_summary).
    """
    messages = [
        {"role": "system", "content": ORCHESTRATOR_SYSTEM},
        {"role": "user",   "content": f"Feature request:\n{feature_request}"},
    ]
    trace_lines: list[str] = []
    sub_agent_outputs: dict[str, str] = {}

    for step in range(1, 15):
        resp   = client.chat.completions.create(model=MODEL, messages=messages, tools=ORCHESTRATOR_TOOLS)
        msg    = resp.choices[0].message
        finish = resp.choices[0].finish_reason

        if finish == "stop":
            trace_lines.append(f"**Done** in {step} step(s).")
            return "\n".join(trace_lines), msg.content

        if finish == "tool_calls":
            messages.append(msg)
            for tc in msg.tool_calls:
                agent_name = tc.function.name
                agent_args = json.loads(tc.function.arguments)

                trace_lines.append(f"**Step {step}:** Orchestrator calls `{agent_name}` sub-agent")

                # ── Dispatch to the sub-agent (which runs its own loop) ────────
                result = SUB_AGENT_MAP[agent_name](agent_args)
                sub_agent_outputs[agent_name] = result

                short = result[:100] + "..." if len(result) > 100 else result
                trace_lines.append(f"  → Sub-agent returned: `{short}`")

                messages.append({"role": "tool", "content": result, "tool_call_id": tc.id})

    return "\n".join(trace_lines), "Max steps reached."

In [ ]:
# ── Gradio UI — Agent as a Tool ──────────────────────────────────────────────
with gr.Blocks(title="A2A: Agent as a Tool") as orchestrator_demo:
    gr.Markdown(
        "## Pattern 5 — Agent as a Tool: Software Delivery Orchestrator\n"
        "The **Orchestrator** dynamically decides which sub-agents to call and in what order.\n"
        "Each sub-agent is a **full agent loop** with its own tools — the Orchestrator treats "
        "them like any other tool call.\n\n"
        "**Sub-agents:** `research_agent` (best practices) · "
        "`coding_agent` (implementation + lint loop) · "
        "`qa_agent` (review + static analysis)"
    )
    with gr.Row():
        with gr.Column(scale=1):
            orch_input = gr.Textbox(
                label="Feature Request",
                placeholder="e.g. Build a connection pool manager for a PostgreSQL client library",
                lines=4
            )
            orch_btn = gr.Button("Run Orchestrator", variant="primary")
        with gr.Column(scale=2):
            orch_trace   = gr.Markdown(label="Orchestrator Dispatch Trace")
            orch_summary = gr.Markdown(label="Delivery Handover Note")

    orch_btn.click(
        fn=run_orchestrator_agent,
        inputs=orch_input,
        outputs=[orch_trace, orch_summary]
    )
    gr.Examples(
        examples=[
            ["Build a connection pool manager for a PostgreSQL database client. Support min/max pool size, connection timeout, and health checks."],
            ["Implement a simple pub/sub event bus in Python. Support multiple subscribers per topic and async event delivery."],
            ["Create a retry decorator for Python functions with configurable max attempts, exponential backoff, and exception filtering."],
        ],
        inputs=orch_input
    )

orchestrator_demo.launch()

---
# Summary

## Pattern Comparison

| Pattern | Topology | Parallelism | Stop Condition | Best For |
|---|---|---|---|---|
| **Handoff** | Linear chain | None — strictly sequential | Last agent completes | Professional pipelines with distinct phases |
| **Supervisor** | Hub-and-spoke | Workers run in parallel | Supervisor finishes synthesis | Multi-dimension evaluation of the same artefact |
| **Peer Debate** | Round-robin + judge | Proposals, critiques, rebuttals all parallel | Judge issues verdict | Architectural decisions with valid competing approaches |
| **Critic / Reflection** | Feedback loop | None — sequential iterations | Score ≥ threshold or max iter | Quality-sensitive outputs that benefit from iteration |
| **Agent as a Tool** | Nested agent loops | Sub-agents run sequentially (can be parallelised) | Orchestrator decides it's done | Complex tasks where sub-problems need their own agent loops |

## When to Use Each Pattern

```
Does the task have a natural sequence of specialist roles?
│
├─ YES, linear (analyst → architect → dev → QA) ────► HANDOFF
│
└─ YES, but specialists can work in parallel
   │
   ├─ Same artefact, different lenses ──────────────── SUPERVISOR
   │   (security + perf + maintainability)
   │
   └─ Competing approaches to the same problem ──────► PEER DEBATE
       (trade-off analysis needed)

Does the output quality need to improve iteratively?
│
└─ YES, and you have clear scoring criteria ─────────► CRITIC / REFLECTION

Are the sub-problems complex enough to need their own agent loops?
│
└─ YES, and the orchestrator should decide dynamically ► AGENT AS A TOOL
```

## A2A vs Single-Agent — Decision Guide

```
Can one LLM with tools solve this?
│
├─ YES ──────────────────────────────────────────────► Use single-agent patterns (Agents.ipynb)
│
└─ NO (needs multiple specialist perspectives)
   │
   ├─ Fixed sequence of specialists ──────────────────► Handoff
   ├─ One coordinator + parallel specialists ──────────► Supervisor
   ├─ Competing solutions + arbitration ───────────────► Peer Debate
   ├─ Iterative quality improvement ──────────────────► Critic / Reflection
   └─ Sub-problems need their own agent loops ─────────► Agent as a Tool
```

## Key Principles

1. **Contracts matter** — define the exact output format each agent produces so the receiving agent can reliably parse it
2. **Parallelise when you can** — `ThreadPoolExecutor` makes independent agent calls free in wall-clock time
3. **Keep agent prompts focused** — each agent should have one clear responsibility; avoid multi-purpose agents
4. **Structure the handoff payload** — strip noise, restructure if needed; pass only what the next agent requires
5. **Always have a stop condition** — score thresholds, max iterations, or explicit termination signals prevent runaway loops
6. **Encapsulate with Agent as a Tool** — when a sub-problem is complex enough, wrap it in its own agent loop and expose it as a tool; the orchestrator doesn't need to know what's inside